In [3]:
import sys, subprocess, pkgutil, io, os, time, random, tempfile, zipfile
from collections import defaultdict
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import torch, inflect, gradio as gr
from transformers import pipeline

# Auto install missing dependencies
required = ["gradio", "transformers", "torch", "timm", "Pillow", "inflect", "matplotlib", "requests", "pandas"]
for pkg in required:
    if pkgutil.find_loader(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# -------------------
# Device selection
# -------------------
device_arg = 0 if torch.cuda.is_available() else -1
device_msg = "Using GPU (CUDA)" if torch.cuda.is_available() else "Using CPU"

# -------------------
# Load models
# -------------------
print("⏳ Loading models (takes time first run)...")
obj_detector = pipeline("object-detection", model="facebook/detr-resnet-50", device=device_arg)
depth_estimator = pipeline("depth-estimation", model="Intel/dpt-hybrid-midas", device=device_arg)
print("✅ Models loaded.", device_msg)

# -------------------
# Utilities
# -------------------
rnd = random.Random(42)
color_map = {}
def get_color_for_label(label):
    if label not in color_map:
        color_map[label] = (rnd.random(), rnd.random(), rnd.random())
    return color_map[label]

p_engine = inflect.engine()

def render_results_in_image(pil_img, predictions, conf_threshold=0.3):
    img = np.array(pil_img).copy()
    fig, ax = plt.subplots()
    ax.imshow(img)
    for pred in predictions:
        if pred["score"] < conf_threshold:
            continue
        box = pred["box"]
        xmin, ymin, xmax, ymax = box.values()
        color = get_color_for_label(pred["label"])
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=color, linewidth=2))
        ax.text(xmin, ymin - 2, f"{pred['label']} ({pred['score']*100:.1f}%)",
                color="white", fontsize=8,
                bbox=dict(facecolor=color, alpha=0.6))
    plt.axis("off")
    buf = io.BytesIO()
    plt.savefig(buf, format="png", bbox_inches="tight", pad_inches=0)
    buf.seek(0)
    out = Image.open(buf).convert("RGB")
    plt.close(fig)
    return out

def format_depth_map(input_img, depth_tensor):
    if isinstance(depth_tensor, torch.Tensor):
        arr = depth_tensor.squeeze().cpu().numpy()
    else:
        arr = np.array(depth_tensor)
    arr = (arr - np.min(arr)) / (np.max(arr) - np.min(arr) + 1e-8)
    cmap = plt.get_cmap("plasma")
    depth_img = (cmap(arr)[:, :, :3] * 255).astype(np.uint8)
    depth_pil = Image.fromarray(depth_img).resize(input_img.size)
    return depth_pil

def summarize_predictions(preds, thr=0.3):
    cnt = defaultdict(int)
    for p in preds:
        if p["score"] >= thr:
            cnt[p["label"]] += 1
    if not cnt:
        return "No confident detections."
    parts = [f"{p_engine.number_to_words(v)} {p_engine.plural_noun(k, v)}" for k, v in cnt.items()]
    return "In this image, there are " + ", ".join(parts) + "."

# -------------------
# Processing function
# -------------------
def process_single_image(img, conf_threshold=0.3):
    start = time.time()
    detections = obj_detector(img)
    det_time = time.time() - start

    filtered = [d for d in detections if d["score"] >= conf_threshold]
    out_img = render_results_in_image(img, detections, conf_threshold)

    start = time.time()
    depth_out = depth_estimator(img)
    depth_img = format_depth_map(img, depth_out["predicted_depth"])
    depth_time = time.time() - start

    total = det_time + depth_time
    fps = 1 / total if total > 0 else 0

    metrics = {
        "Detection Time (s)": round(det_time, 3),
        "Depth Time (s)": round(depth_time, 3),
        "Total Time (s)": round(total, 3),
        "FPS": round(fps, 2),
        "Num Detections": len(filtered),
        "Avg Confidence": round(np.mean([f["score"] for f in filtered]) if filtered else 0, 3)
    }
    summary = summarize_predictions(detections, conf_threshold)
    return out_img, depth_img, summary, "\n".join([f"{k}: {v}" for k, v in metrics.items()])

# -------------------
# Gradio UI
# -------------------
title_md = """
<div style="text-align:center;">
  <h2>🎯 Object Detection + Depth Estimation (DETR + MiDaS)</h2>
  <p>Real-time detection with performance metrics (FPS, timing, confidence).</p>
</div>
"""

with gr.Blocks() as demo:
    gr.Markdown(title_md)

    with gr.Row():
        with gr.Column():
            img_in = gr.Image(type="pil", label="Upload Image")
            conf = gr.Slider(0, 1, 0.3, 0.01, label="Confidence Threshold")
            run_btn = gr.Button("Run Detection & Depth")
        with gr.Column():
            img_out = gr.Image(label="Detection Output")
            depth_out = gr.Image(label="Depth Map")
            summary = gr.Textbox(label="Summary")
            metrics = gr.Textbox(label="Performance Metrics", lines=6)

    def run_pipeline(img, conf):
        if img is None:
            return None, None, "Please upload an image!", ""
        det, dep, sumtxt, perf = process_single_image(img, conf)
        return det, dep, sumtxt, perf

    run_btn.click(run_pipeline, inputs=[img_in, conf], outputs=[img_out, depth_out, summary, metrics])

    gr.Markdown(f"**Note:** {device_msg}. First run may take time while downloading model weights.")

demo.launch(share=True)

/tmp/ipython-input-1776049514.py:14: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader(pkg) is None:


⏳ Loading models (takes time first run)...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pas

✅ Models loaded. Using GPU (CUDA)
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0b6d60d5f5cd57f366.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
